# 06 Regression Test Harness Design (DSPy, 2026)

## What This Lesson Is
Build regression checks so prompt/program changes do not silently degrade behavior.

## Scientific Lens
- Concept: Behavioral regression control
- Measure: Golden test pass rate over versioned test set
- Validity Limit: Goldens must be refreshed or they become stale and misleading.


## How It Works
1. Define deterministic golden tests.
2. Run pass/fail assertions.
3. Apply live outputs against same acceptance checks.


In [ ]:
import os
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("Regression harness lesson preflight complete")


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
def normalize(text: str) -> str:
    return " ".join(text.strip().lower().split())

goldens = [
    {"in": " Error: Timeout ", "out": "error: timeout"},
    {"in": "DEPLOY SUCCEEDED", "out": "deploy succeeded"},
]

for g in goldens:
    got = normalize(g["in"])
    print(g["in"], "->", got)
    assert got == g["out"]


In [ ]:
# Live Demo
import os

try:
    import dspy
except Exception as exc:
    print(f"Skipping live regression demo: dspy unavailable ({exc})")
else:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("Skipping live regression demo: OPENAI_API_KEY not set.")
    else:
        dspy.configure(lm=dspy.LM("openai/gpt-4.1-mini", api_key=api_key, temperature=0))
        predict = dspy.Predict("question -> answer")
        tests = [
            "Define idempotency in one sentence.",
            "Define fallback routing in one sentence.",
        ]
        for t in tests:
            out = predict(question=t)
            print(t, "=>", out.answer)
            assert out.answer


## Applied Labs
1. Add semantic assertion checks beyond non-empty string validation.
2. Version goldens and compare against previous release baseline.
3. Fail the notebook intentionally and verify clear failure diagnostics.

## Validation Checklist
- Goldens are explicit and deterministic.
- Assertion failures identify which case regressed.
- Live outputs are evaluated with repeatable acceptance criteria.

## Further Reading
- [Google Testing Blog - Goldens](https://testing.googleblog.com/2014/07/testing-on-toilet-dont-overuse-mocks.html)
- [DSPy Evaluation](https://dspy.ai/learn/evaluation/overview/)
- [Software Regression Testing](https://en.wikipedia.org/wiki/Regression_testing)
